# Adding Metadata | Cleaning the enriched dataset | Selecting the corpus

The following python code walks through the proces of creating a balanced dataset for stylometric anlysis. The names of the authors have already been standardized in OpenRefine. Metadata such a the *publication date* and *genre* of the novel,  as well as the *gender* and *country of citizenship* have been reconciled against Wikidata using OpenRefine. The Downloads column represents the number of downloads from the Project Gutenberg webiste, and will be the basis for the selection of novels, alongside country of citizenship.

The goal is to standardize genre and country of citizenship, and to make a balanced selection of novels

In [1]:
import pandas as pd

df = pd.read_csv("sf-fantasy-novels-csv-enriched.csv")

df.head(10)

,Title,genre,publication date,Author,country of citizenship,sex or gender,Downloads,Plain Text UTF-8 Link
0,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...",1818-01-01T00:00:00Z,Mary Shelley,"United Kingdom of Great Britain and Ireland, K...",female,264413,https://www.gutenberg.org/cache/epub/84/pg84.txt
1,The King in Yellow,"Decadent movement, horror literature, supernat...",1895-01-01T00:00:00Z,Robert W. Chambers,United States,male,68842,https://www.gutenberg.org/cache/epub/8492/pg84...
2,Dracula,"epistolary novel, invasion literature, Gothic ...",1897-05-26T00:00:00Z,Bram Stoker,United Kingdom of Great Britain and Ireland,male,42071,https://www.gutenberg.org/cache/epub/345/pg345...
3,The Wonderful Wizard of Oz,"children's story, fantasy",1900-01-01T00:00:00Z,L. Frank Baum,United States,male,24328,https://www.gutenberg.org/cache/epub/55/pg55.txt
4,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...",1818-01-01T00:00:00Z,Mary Shelley,"United Kingdom of Great Britain and Ireland, K...",female,20274,https://www.gutenberg.org/cache/epub/41445/pg4...
5,Gulliver's Travels,"satire, fantasy, sea adventure novel",1726-10-28T00:00:00Z,Jonathan Swift,Kingdom of Ireland,male,17684,https://www.gutenberg.org/cache/epub/829/pg829...
6,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...",1818-01-01T00:00:00Z,Mary Shelley,"United Kingdom of Great Britain and Ireland, K...",female,14471,https://www.gutenberg.org/cache/epub/42324/pg4...
7,Carmilla,"fantasy, lesbian literature, vampire fiction, ...",1872-01-01T00:00:00Z,Sheridan Le Fanu,"United Kingdom of Great Britain and Ireland, B...",male,12860,https://www.gutenberg.org/cache/epub/10007/pg1...
8,Rip Van Winkle,time-travel fiction,1818-01-01T00:00:00Z,Washington Irving,United States,male,11659,https://www.gutenberg.org/cache/epub/60976/pg6...
9,Twenty Thousand Leagues Under the Sea,"science fiction, nautical fiction, scientific ...",1869-03-20T00:00:00Z,Jules Verne,France,male,10353,https://www.gutenberg.org/cache/epub/164/pg164...


### Standardizing country of citizenship

The United Kingdom has had multiple names over history:
- United Kingdom
- United Kingdom of Great Britain and Ireland
- United Kingdom of Great Britain
- British Empire

The reconciled values from wikidata have to be standardized into "United Kingdom" for the purpose of later analysis. The following function standardizes these values. This is not neccesary for the United States: these values are already consistant. We are collecting a corpus of US and UK authored novels, so we can ignore other nationalities.

In [2]:
def standardize_nationality(column):
    """
    Takes the DataFrame and standardizes UK values
    """
    uk_variants = ["United Kingdom","United Kingdom of Great Britain and Ireland","United Kingdom of Great Britain","British Empire"]
    
    for variant in uk_variants:
        if isinstance(column, str):  #error handling for null values
            if variant in column:
                return "United Kingdom"

    return column #keeps old values in case of no match

df["country of citizenship"] = df["country of citizenship"].apply(standardize_nationality)
df.head(10)

,Title,genre,publication date,Author,country of citizenship,sex or gender,Downloads,Plain Text UTF-8 Link
0,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...",1818-01-01T00:00:00Z,Mary Shelley,United Kingdom,female,264413,https://www.gutenberg.org/cache/epub/84/pg84.txt
1,The King in Yellow,"Decadent movement, horror literature, supernat...",1895-01-01T00:00:00Z,Robert W. Chambers,United States,male,68842,https://www.gutenberg.org/cache/epub/8492/pg84...
2,Dracula,"epistolary novel, invasion literature, Gothic ...",1897-05-26T00:00:00Z,Bram Stoker,United Kingdom,male,42071,https://www.gutenberg.org/cache/epub/345/pg345...
3,The Wonderful Wizard of Oz,"children's story, fantasy",1900-01-01T00:00:00Z,L. Frank Baum,United States,male,24328,https://www.gutenberg.org/cache/epub/55/pg55.txt
4,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...",1818-01-01T00:00:00Z,Mary Shelley,United Kingdom,female,20274,https://www.gutenberg.org/cache/epub/41445/pg4...
5,Gulliver's Travels,"satire, fantasy, sea adventure novel",1726-10-28T00:00:00Z,Jonathan Swift,Kingdom of Ireland,male,17684,https://www.gutenberg.org/cache/epub/829/pg829...
6,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...",1818-01-01T00:00:00Z,Mary Shelley,United Kingdom,female,14471,https://www.gutenberg.org/cache/epub/42324/pg4...
7,Carmilla,"fantasy, lesbian literature, vampire fiction, ...",1872-01-01T00:00:00Z,Sheridan Le Fanu,United Kingdom,male,12860,https://www.gutenberg.org/cache/epub/10007/pg1...
8,Rip Van Winkle,time-travel fiction,1818-01-01T00:00:00Z,Washington Irving,United States,male,11659,https://www.gutenberg.org/cache/epub/60976/pg6...
9,Twenty Thousand Leagues Under the Sea,"science fiction, nautical fiction, scientific ...",1869-03-20T00:00:00Z,Jules Verne,France,male,10353,https://www.gutenberg.org/cache/epub/164/pg164...


### Standardizing Genre
The data in this dataframe was scraped from the Science Fiction & Fantasy category available on Project Gutenberg. However, to classify the genres more strictly, the genre column in the dataframe contains reconciled Wikidata values of the novels. for the scope of this research, we want to strictly look at Fanatasy novels, Science Fiction novels or a combination of both. Therefore, we have to standardize the wikidata genres in the code below.

In [3]:
def categorize_genre(genre_column):
    """
    Takes a string (wikidata genres) and returns a standardized genre:
    'science fiction', 'fantasy', 'both', or '' if neither.
    """
    # Ensure the genre string is lowercase for consistent matching
    genre_lower = str(genre_column).lower()
    
    has_scifi = "science fiction" in genre_lower    #boolean
    has_fantasy = "fantasy" in genre_lower          #boolean

    if has_scifi and has_fantasy:
        return "fantasy, science fiction"
    elif has_scifi:
        return "science fiction"
    elif has_fantasy:
        return "fantasy"
    else:
        return None  # None is recognized by pandas .dropna() which will be used later

In [4]:
stand_genres = df['genre'].apply(categorize_genre)

df.insert(2, "standardized genre", stand_genres)

In [5]:
df.head(2)

,Title,genre,standardized genre,publication date,Author,country of citizenship,sex or gender,Downloads,Plain Text UTF-8 Link
0,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...","fantasy, science fiction",1818-01-01T00:00:00Z,Mary Shelley,United Kingdom,female,264413,https://www.gutenberg.org/cache/epub/84/pg84.txt
1,The King in Yellow,"Decadent movement, horror literature, supernat...",None,1895-01-01T00:00:00Z,Robert W. Chambers,United States,male,68842,https://www.gutenberg.org/cache/epub/8492/pg84...


### Filtering country of citizenship
Here we alter the dataframe to only include US and UK authors

In [6]:
uk_df = df[df["country of citizenship"] == "United Kingdom"]
us_df = df[df["country of citizenship"] == "United States"]

df = pd.concat([uk_df, us_df], ignore_index=True)
df = df.sort_values("Downloads", ascending=False).reset_index(drop=True)  #Here we sort again on based Downloads and reset the index of the df
df.head()

,Title,genre,standardized genre,publication date,Author,country of citizenship,sex or gender,Downloads,Plain Text UTF-8 Link
0,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...","fantasy, science fiction",1818-01-01T00:00:00Z,Mary Shelley,United Kingdom,female,264413,https://www.gutenberg.org/cache/epub/84/pg84.txt
1,The King in Yellow,"Decadent movement, horror literature, supernat...",None,1895-01-01T00:00:00Z,Robert W. Chambers,United States,male,68842,https://www.gutenberg.org/cache/epub/8492/pg84...
2,Dracula,"epistolary novel, invasion literature, Gothic ...",None,1897-05-26T00:00:00Z,Bram Stoker,United Kingdom,male,42071,https://www.gutenberg.org/cache/epub/345/pg345...
3,The Wonderful Wizard of Oz,"children's story, fantasy",fantasy,1900-01-01T00:00:00Z,L. Frank Baum,United States,male,24328,https://www.gutenberg.org/cache/epub/55/pg55.txt
4,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...","fantasy, science fiction",1818-01-01T00:00:00Z,Mary Shelley,United Kingdom,female,20274,https://www.gutenberg.org/cache/epub/41445/pg4...


### Filtering time period

In [7]:
#this codes changes the datatype of the publication date column to datetime
df['publication date'] = pd.to_datetime(df['publication date'], errors='coerce', utc=True)
df['publication date'].dtype

datetime64[ns, UTC]

The function below converts the datetime object to integers representing the publication year for standardization purposes for analysis in Stylo

In [8]:
import re

def extract_year(date_str):
    match = re.search(r'\b(18|19|20)\d{2}\b', str(date_str))
    return int(match.group()) if match else None

df['publication date'] = df['publication date'].apply(extract_year)
df['publication date'] = df['publication date'].astype("Int64") #convert float to int
df

,Title,genre,standardized genre,publication date,Author,country of citizenship,sex or gender,Downloads,Plain Text UTF-8 Link
0,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...","fantasy, science fiction",1818,Mary Shelley,United Kingdom,female,264413,https://www.gutenberg.org/cache/epub/84/pg84.txt
1,The King in Yellow,"Decadent movement, horror literature, supernat...",None,1895,Robert W. Chambers,United States,male,68842,https://www.gutenberg.org/cache/epub/8492/pg84...
2,Dracula,"epistolary novel, invasion literature, Gothic ...",None,1897,Bram Stoker,United Kingdom,male,42071,https://www.gutenberg.org/cache/epub/345/pg345...
3,The Wonderful Wizard of Oz,"children's story, fantasy",fantasy,1900,L. Frank Baum,United States,male,24328,https://www.gutenberg.org/cache/epub/55/pg55.txt
4,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...","fantasy, science fiction",1818,Mary Shelley,United Kingdom,female,20274,https://www.gutenberg.org/cache/epub/41445/pg4...
...,...,...,...,...,...,...,...,...,...
2980,Nahkaratti (Finnish),NaN,None,<NA>,Arthur Conan Doyle,United Kingdom,male,166,https://www.gutenberg.org/cache/epub/56573/pg5...
2981,Cognitive Behavioral Therapy Treatment for a W...,NaN,None,2019,William Campbell Gault,United States,male,165,https://www.gutenberg.org/cache/epub/74139/pg7...
2982,Punainen rutto (Finnish),NaN,None,<NA>,Jack London,United States,male,164,https://www.gutenberg.org/cache/epub/56785/pg5...
2983,The Reluctant Weapon,NaN,None,<NA>,Howard L. Myers,United States,male,164,https://www.gutenberg.org/cache/epub/31995/pg3...


In [9]:
# #Create a seelction of 19th century novels
df = df[(df['publication date'] >= 1880) & (df['publication date'] < 1980)]

#filter out all novels that are not classified as fiction, science fiction or both
df = df.dropna(subset=['standardized genre']) 

#check for dupicate novels titles (this happens sometimes on Project GUtenberg)
# df.value_counts(subset=['Title','Author']).head()

#### As we can see, there are some duplicates which we should remove.

In [10]:
df.shape

(330, 9)

In [11]:
df = df.drop_duplicates(subset=['Title','Author'])  
df.value_counts(subset=['Title','Author'])

Title                             Author              
"Let There Be Light"              H. B. Fyfe              1
The Island of Dr Moreau           H. G. Wells             1
The Man Who Saved the Earth       Austin Hall             1
The Magical Mimics in Oz          Jack Snow               1
The Magic of Oz                   L. Frank Baum           1
                                                         ..
Pattern for Conquest              George O. Smith         1
Palos of the Dog Star Pack        John Ulrich Giesy       1
Ozoplaning with the Wizard of Oz  Ruth Plumly Thompson    1
Ozma of Oz                        L. Frank Baum           1
Worlds of the Imperium            Keith Laumer            1
Name: count, Length: 294, dtype: int64

#### To prevent high similarity between novels by the same author we limit the dataset to 1 novel per author. This way, we isolate genre en country of citizenship as factors for stylistic similarity. The first novel for eacht other in the df (with the highest amount of downloads) is kept.


In [12]:
df = df.drop_duplicates(subset=['Author'])

#### In the code below we see how many novels are left per category after all the previous filtering of the corpus.

In [13]:
df.value_counts(['country of citizenship','standardized genre'])

country of citizenship  standardized genre      
United States           science fiction             66
United Kingdom          fantasy                     16
United States           fantasy                     15
United Kingdom          science fiction             12
United States           fantasy, science fiction     6
Name: count, dtype: int64

#### As we can see, the category with the lowest amount of novels is Science Fiction by UK authors with 12 novels. We will limit the other categories to 12 novels as well, so that the dataset is balanced. (novels classified as both sci-fi and fantasy will be left out, since they cannot be categorized clearly)

In [14]:
#US Sci-Fi dataframe (selects the first twelve novels with the highest downloads)
us_scifi = df[(df['country of citizenship'] == 'United States') & (df['standardized genre'] == 'science fiction')].head(12).reset_index(drop=True)

#UK sci-fi dataframe (selects the first twelve with the highest downloads)
uk_scifi = df[(df['country of citizenship'] == 'United Kingdom') & (df['standardized genre'] == 'science fiction')].head(12).reset_index(drop=True)

#US Fantasy dataframe (selects the first twelve with the highest downloads)
us_fantasy = df[(df['country of citizenship'] == 'United States') & (df['standardized genre'] == 'fantasy')].head(12).reset_index(drop=True)

#UK Fantasy dataframe (selects the first twelve with the highest downloads)
uk_fantasy = df[(df['country of citizenship'] == 'United Kingdom') & (df['standardized genre'] == 'fantasy')].head(12).reset_index(drop=True)

In [15]:
us_fantasy

,Title,genre,standardized genre,publication date,Author,country of citizenship,sex or gender,Downloads,Plain Text UTF-8 Link
0,The Wonderful Wizard of Oz,"children's story, fantasy",fantasy,1900,L. Frank Baum,United States,male,24328,https://www.gutenberg.org/cache/epub/55/pg55.txt
1,Red Nails,"sword and sorcery, fantasy",fantasy,1936,Robert E. Howard,United States,male,2140,https://www.gutenberg.org/cache/epub/32759/pg3...
2,David and the Phoenix,fantasy,fantasy,1957,Edward Ormondroyd,United States,male,937,https://www.gutenberg.org/cache/epub/27922/pg2...
3,Jack Pumpkinhead of Oz,fantasy,fantasy,1929,Ruth Plumly Thompson,United States,female,530,https://www.gutenberg.org/cache/epub/75720/pg7...
4,The Blue Star,fantasy,fantasy,1969,Fletcher Pratt,United States,male,519,https://www.gutenberg.org/cache/epub/56889/pg5...
5,"Jurgen, A Comedy of Justice",fantasy,fantasy,1919,James Branch Cabell,United States,male,459,https://www.gutenberg.org/cache/epub/8771/pg87...
6,Davy and the Goblin,fantasy,fantasy,1885,Charles E. Carryl,United States,male,446,https://www.gutenberg.org/cache/epub/25031/pg2...
7,Mr. Munchausen,fantasy,fantasy,1901,John Kendrick Bangs,United States,male,411,https://www.gutenberg.org/cache/epub/33432/pg3...
8,The Return of Tharn,fantasy,fantasy,1956,Howard Browne,United States,male,359,https://www.gutenberg.org/cache/epub/33529/pg3...
9,The Magical Mimics in Oz,fantasy,fantasy,1946,Jack Snow,United States,male,351,https://www.gutenberg.org/cache/epub/56555/pg5...


### Now that we have 4 dataframes representing the four categories, we can scrape the txt files from Project Gutenberg

In [21]:
import requests
import os

#UK fantasy txt scraping (this will overwrite any existing files in the repo!)
for idx, row in uk_fantasy.iterrows():   
    author = row['Author']
    url = row['Plain Text UTF-8 Link']

    response = requests.get(url)

    filename = f"{author}.txt"
    filepath = os.path.join("data/uk_fantasy", filename)
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(response.text)

In [17]:
#UK scifi txt scraping
for idx, row in uk_scifi.iterrows():   
    author = row['Author']
    url = row['Plain Text UTF-8 Link']

    response = requests.get(url)

    filename = f"{author}.txt"
    filepath = os.path.join("data/uk_scifi", filename)
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(response.text)

In [18]:
#US scifi txt scraping
for idx, row in us_scifi.iterrows():   
    author = row['Author']
    url = row['Plain Text UTF-8 Link']

    response = requests.get(url)

    filename = f"{author}.txt"
    filepath = os.path.join("data/us_scifi", filename)
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(response.text)

In [20]:
#US fantasy txt scraping
for idx, row in us_fantasy.iterrows():   
    author = row['Author']
    url = row['Plain Text UTF-8 Link']

    response = requests.get(url)

    filename = f"{author}.txt"
    filepath = os.path.join("data/us_fantasy", filename)
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(response.text)